<a href="https://colab.research.google.com/github/Alessandro-json/AI_PostProcessing_Detection/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 2 - Joint Detection of AI-Generated Images and Post-Processing Alterations in Real-World Scenarios

# Imports

In [3]:
import os
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


# Globals


In [4]:
# Choose where the project will be stored in Colab.
WORKDIR = Path('/content')

REPO_URL = 'https://github.com/Alessandro-json/AI_PostProcessing_Detection'

# Repository folder name after git clone.
REPO_DIR = WORKDIR / 'REPO'

# Main paths used by the scripts.
TRAIN_CSV = 'data/splits/train_balanced.csv'
VAL_CSV = 'data/splits/val_balanced.csv'
TEST_CSV = 'data/splits/test_balanced.csv'

IMAGE_ROOT = "/content/data/raw/RRDataset_subset"
CHECKPOINT_NAME = 'best_rgb.pt'
CHECKPOINT_PATH = f'checkpoints/{CHECKPOINT_NAME}'
OUTPUT_DIR = 'results/rgb_baseline'
DEPTH_ROOT = "/content/drive/MyDrive/CV_Project/depth_maps"

DATASET_FILE_ID = "1HJamsAB4Lj90xNjA6tkfIrstHGJqklsI"
DATASET_ZIP_PATH = "/content/RRDataset_subset.zip"

# Training hyperparameters for the first baseline.
EPOCHS = 10
BATCH_SIZE = 32
IMAGE_SIZE = 224
NUM_WORKERS = 2
LEARNING_RATE = 1e-4

# Multi-task loss weights.
LAMBDA_FAKE = 1.0
LAMBDA_TRANSFORM = 1.0


# Setup repository



In [5]:
%cd {WORKDIR}

if REPO_DIR.exists():
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}


/content
Cloning into '/content/REPO'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 51 (delta 16), reused 27 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 207.21 KiB | 7.67 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/REPO


# Install dependencies

In [6]:
# Install project dependencies.
!pip install -r requirements.txt


In [7]:
!pip install -q gdown

# Utils

In [8]:
def show_csv_summary(csv_path):
    """Print a quick summary of one split CSV."""
    path = Path(csv_path)
    if not path.exists():
        print(f'Missing file: {path}')
        return None

    df = pd.read_csv(path)
    print(f'File: {path}')
    print(f'Rows: {len(df)}')
    print('Columns:', list(df.columns))

    if 'fake_label' in df.columns:
        print('\nFake label distribution:')
        print(df['fake_label'].value_counts().sort_index())

    if 'transform_label' in df.columns:
        print('\nTransform label distribution:')
        print(df['transform_label'].value_counts().sort_index())

    if {'fake_label', 'transform_label'}.issubset(df.columns):
        print('\nJoint distribution:')
        print(pd.crosstab(df['transform_label'], df['fake_label'], rownames=['transform'], colnames=['fake']))

    return df


def show_image_exists_check(df, image_root, n=5):
    """Check whether the first n image paths exist."""
    if df is None:
        return

    root = Path(image_root)
    print(f'Image root: {root}')

    for rel_path in df['image_path'].head(n):
        full_path = root / rel_path
        print(full_path, 'OK' if full_path.exists() else 'MISSING')


# Data


In [9]:
train_df = show_csv_summary(TRAIN_CSV)


File: data/splits/train_balanced.csv
Rows: 2100
Columns: ['image_path', 'fake_label', 'transform_label', 'category', 'transform_name', 'fake_name']

Fake label distribution:
fake_label
0    1050
1    1050
Name: count, dtype: int64

Transform label distribution:
transform_label
0    700
1    700
2    700
Name: count, dtype: int64

Joint distribution:
fake         0    1
transform          
0          350  350
1          350  350
2          350  350


In [10]:
val_df = show_csv_summary(VAL_CSV)


File: data/splits/val_balanced.csv
Rows: 450
Columns: ['image_path', 'fake_label', 'transform_label', 'category', 'transform_name', 'fake_name']

Fake label distribution:
fake_label
0    225
1    225
Name: count, dtype: int64

Transform label distribution:
transform_label
0    150
1    150
2    150
Name: count, dtype: int64

Joint distribution:
fake        0   1
transform        
0          75  75
1          75  75
2          75  75


In [11]:
test_df = show_csv_summary(TEST_CSV)


File: data/splits/test_balanced.csv
Rows: 450
Columns: ['image_path', 'fake_label', 'transform_label', 'category', 'transform_name', 'fake_name']

Fake label distribution:
fake_label
0    225
1    225
Name: count, dtype: int64

Transform label distribution:
transform_label
0    150
1    150
2    150
Name: count, dtype: int64

Joint distribution:
fake        0   1
transform        
0          75  75
1          75  75
2          75  75


In [12]:
!gdown --id "$DATASET_FILE_ID" -O "$DATASET_ZIP_PATH"

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1HJamsAB4Lj90xNjA6tkfIrstHGJqklsI
From (redirected): https://drive.google.com/uc?id=1HJamsAB4Lj90xNjA6tkfIrstHGJqklsI&confirm=t&uuid=d29a2819-59ed-4fef-b9ed-fdd7947a6f2d
To: /content/RRDataset_subset.zip
100% 1.27G/1.27G [00:17<00:00, 71.7MB/s]


In [13]:
!rm -rf /content/data/raw/RRDataset_subset
!mkdir -p /content/data/raw
!unzip -n -q "$DATASET_ZIP_PATH" -d /content/data/raw

In [14]:
show_image_exists_check(train_df, IMAGE_ROOT, n=5)


Image root: /content/data/raw/RRDataset_subset
/content/data/raw/RRDataset_subset/original/real/real_006970.jpg OK
/content/data/raw/RRDataset_subset/original/real/real_003543.jpg OK
/content/data/raw/RRDataset_subset/original/real/real_004687.jpg OK
/content/data/raw/RRDataset_subset/original/real/real_001434.jpg OK
/content/data/raw/RRDataset_subset/original/real/real_004760.jpg OK


# Train

This cell launches training for the RGB multi-task baseline.


In [15]:
!python src/train.py \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --checkpoint_name {CHECKPOINT_NAME} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --lr {LEARNING_RATE} \
  --lambda_fake {LAMBDA_FAKE} \
  --lambda_transform {LAMBDA_TRANSFORM}


Traceback (most recent call last):
  File "/content/REPO/src/train.py", line 9, in <module>
    from dataset import RRDatasetFromCSV, build_train_transform, build_eval_transform
  File "/content/REPO/src/dataset.py", line 3, in <module>
    import pandas as pd
  File "/usr/local/lib/python3.12/dist-packages/pandas/__init__.py", line 49, in <module>
    from pandas.core.api import (
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/api.py", line 47, in <module>
    from pandas.core.groupby import (
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/groupby/__init__.py", line 1, in <module>
    from pandas.core.groupby.generic import (
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/groupby/generic.py", line 68, in <module>
    from pandas.core.frame import DataFrame
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/frame.py", line 149, in <module>
    from pandas.core.generic import (
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/ge

# Evaluation

This cell evaluates the best checkpoint and saves:

- `predictions.csv`;
- `metrics.json`;
- `confusion_fake.png`;
- `confusion_transform.png`.


In [16]:
!python src/evaluate.py \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --checkpoint {CHECKPOINT_PATH} \
  --output_dir {OUTPUT_DIR} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}


Traceback (most recent call last):
  File "/content/REPO/src/evaluate.py", line 7, in <module>
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 3001, in <module>
    _import_device_backends()
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 2949, in _import_device_backends
    backend_extensions = entry_points(group=group_name)
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/metadata/__init__.py", line 913, in entry_points
    return EntryPoints(eps).select(**params)
           ^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/metadata/__init__.py", line 911, in <genexpr>
    dist.entry_points for dist in _unique(distributions())
    ^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/metadata/__init__.py", line 471, in entry_points
    return EntryPoints._from_text_for(self.read_text('entry_points.txt'), self)
                                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
 

# Results

In [17]:
metrics_path = Path(OUTPUT_DIR) / 'metrics.json'

if metrics_path.exists():
    with open(metrics_path, 'r', encoding='utf-8') as f:
        metrics = json.load(f)
    print(json.dumps(metrics, indent=4))
else:
    print(f'Metrics file not found: {metrics_path}')


Metrics file not found: results/rgb_baseline/metrics.json


In [18]:
predictions_path = Path(OUTPUT_DIR) / "predictions.csv"

if predictions_path.exists():
    predictions_df = pd.read_csv(predictions_path)
else:
    print(f"Predictions file not found: {predictions_path}")

Predictions file not found: results/rgb_baseline/predictions.csv


In [19]:
print("True fake labels:")
print(predictions_df["true_fake"].value_counts())

print("\nPredicted fake labels:")
print(predictions_df["pred_fake"].value_counts())

print("\nTrue transform labels:")
print(predictions_df["true_transform"].value_counts())

print("\nPredicted transform labels:")
print(predictions_df["pred_transform"].value_counts())

True fake labels:


NameError: name 'predictions_df' is not defined

In [ ]:
# Show saved confusion matrices if they exist.
for filename in ['confusion_fake.png', 'confusion_transform.png']:
    path = Path(OUTPUT_DIR) / filename
    if path.exists():
        img = plt.imread(path)
        plt.figure(figsize=(7, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.title(filename)
        plt.show()
    else:
        print(f'Missing: {path}')


#DEPTH

In [20]:
!find /content -name "real_006970.jpg"

/content/data/raw/RRDataset_subset/original/real/real_006970.jpg


##Depth map generation

###first small debug

In [ ]:
!python src/generate_depth_map.py \
  --csv_paths {TRAIN_CSV} {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --model_type MiDaS_small \
  --max_images 5

###full depth map generation

In [ ]:
!python src/generate_depth_map.py \
  --csv_paths {TRAIN_CSV} {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --model_type MiDaS_small

##First try with depth only

In [ ]:
!python src/train_depth.py \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint_name {CHECKPOINT_NAME} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --lr {LEARNING_RATE} \
  --lambda_fake {LAMBDA_FAKE} \
  --lambda_transform {LAMBDA_TRANSFORM}

##Second try with also edge consistency

In [ ]:
CHECKPOINT_NAME = "best_depth_only.pt"

!python src/train_depth.py \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint_name {CHECKPOINT_NAME} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --lr {LEARNING_RATE} \
  --lambda_fake {LAMBDA_FAKE} \
  --lambda_transform {LAMBDA_TRANSFORM} \
  --no_edge